In [ ]:
SEED = 0
N = 16
DRIVE_ROOT = "/content/drive/MyDrive/cs182-sweep"
TARGET_MANIFEST = None  # Set to the same manifest path used for training if you supplied one.

# Evaluate one saved LoRA adapter
Choose a GPU runtime, set the seed and training example count above, then run all cells. The notebook evaluates every configured benchmark with one GPU worker. Run different pairs in separate runtimes. Use one active runtime per pair so each results file has a single writer.

On reconnect, select the same pair. The notebook restores completed rows from Drive and snapshots progress every minute and when the worker exits.

In [ ]:
REVISION = "38daf7cd68699438c0a5b6f6400877572f4c0137"
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import tomllib
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path(DRIVE_ROOT)
WORK = Path("/content/cs182-lora-evaluation")
WORK.mkdir(parents=True, exist_ok=True)
repo_url = "https://github.com/KG2468/ICLvsLoRA.git"
repo = WORK / "repo"
if not repo.exists():
    source = DRIVE / "repo"
    subprocess.run(["git", "clone", "--no-checkout", str(source) if (source / ".git").exists() else repo_url, str(repo)], check=True)
if subprocess.run(["git", "-C", str(repo), "cat-file", "-e", REVISION + "^{commit}"], capture_output=True).returncode:
    subprocess.run(["git", "-C", str(repo), "fetch", repo_url, REVISION], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", REVISION], check=True)
manifest = json.loads(Path(TARGET_MANIFEST).read_text()) if TARGET_MANIFEST else None
cfg = manifest["config"] if manifest else tomllib.loads((repo / "configs/full.toml").read_text())
exp = cfg["experiment"]

if type(SEED) is not int or SEED not in exp["seeds"] or type(N) is not int or N not in exp["sizes"]:
    raise ValueError(f"Choose SEED from {exp['seeds']} and N from {exp['sizes']}")
label = f"seed{SEED}-n{N}"
run_dir = Path(exp["output_dir"]) / exp["name"]
model_dir = Path(cfg["model"]["directory"]) / cfg["model"]["precision"]
data_dir = Path(exp["data_dir"])
for path in (run_dir, model_dir, data_dir):
    if path.is_absolute() or ".." in path.parts:
        raise ValueError(f"Expected a project-relative path: {path}")
source = DRIVE / run_dir / "adapters" / label
adapter = Path(tempfile.mkdtemp(prefix=label + "-", dir=WORK))
for name in ("complete.json", "adapter_config.json", "adapters.safetensors"):
    if not (source / name).is_file():
        raise FileNotFoundError(f"Missing checkpoint file: {source / name}. Check the seed, N and Drive folder.")
    shutil.copy2(source / name, adapter / name)
def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()
marker = json.loads((adapter / "complete.json").read_text())
if sha256(adapter / "adapters.safetensors") != marker["weights_sha256"]:
    raise ValueError(f"Adapter checksum mismatch: {source}")
print("Staged checkpoint:", source)

## Set up CUDA, Python and the HumanEval scorer

In [ ]:
import runpy
from getpass import getpass
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Choose a Colab NVIDIA GPU runtime and rerun.")
details = subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version,compute_cap",
                                   "--format=csv,noheader"], text=True).splitlines()[0]
name, driver, capability = [part.strip() for part in details.split(",")]
if float(capability) < 7.5 or tuple(map(int, driver.split("."))) < (550, 54, 14):
    raise RuntimeError(f"Unsupported NVIDIA GPU or driver: {details}")
DEVICE = "cuda"
MLX = "mlx[cuda13]==0.32.4" if int(driver.split(".")[0]) >= 580 else "mlx[cuda12]==0.32.4"
print(details)
if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
UV = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
VENV = WORK / ("venv-" + DEVICE)
subprocess.run([UV, "python", "install", "3.12.11"], check=True)
if not (VENV / "bin/python").exists():
    subprocess.run([UV, "venv", "--python", "3.12.11", str(VENV)], check=True)
PY = str(VENV / "bin/python")
subprocess.run([UV, "pip", "install", "--python", PY, MLX, "mlx-lm[train]==0.32.2",
                "datasets==4.8.5", "transformers==5.23.0", "huggingface-hub==1.33.0",
                "matplotlib==3.11.2"], check=True)
env = {**os.environ, "PYTHONPATH": str(repo / "src"), "PYTHONUNBUFFERED": "1"}
def run_python(*args, **kwargs):
    return subprocess.run([PY, *args], env=env, cwd=WORK, check=True, **kwargs)
run_python("-c", "import mlx.core as mx; mx.set_default_device(mx.gpu); "
                 "print(mx.device_info()); print('GPU check:', (mx.ones((2,2)) @ mx.ones((2,2))).tolist())")
if DEVICE == "cuda":
    import ctypes.util
    if not ctypes.util.find_library("seccomp"):
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "libseccomp2"], check=True)
    run_python("-c", "from final_project.scoring import execute; "
               "r=execute('import math\\ndef f(): return math.factorial(4)', "
               "'def check(f): assert f()==24', 'f', 10); "
               "assert r['passed'], r; print('Linux scoring check passed')")
    # Exercise the isolation boundary before giving the scorer model output.
    probe = """import os, socket
def f():
    blocked = 0
    for action in (lambda: open('/etc/passwd'), lambda: socket.socket(), lambda: os.fork()):
        try:
            action()
        except OSError:
            blocked += 1
    return blocked
"""
    run_python("-c", "from final_project.scoring import execute; "
               + "r=execute(" + repr(probe) + ", 'def check(f): assert f()==3', 'f', 10); "
               + "assert r['passed'], r")
storage = runpy.run_path(str(repo / "scripts/colab_storage.py"))
settings = WORK / "target-config.json"
settings.write_text(json.dumps(cfg))
load_cfg = "import json, sys; cfg = json.load(open(sys.argv[1])); "

## Stage benchmark data and model weights on the local disk

In [ ]:
storage["copy_tree_once"](DRIVE / data_dir, WORK / data_dir)
dataset_manifest = WORK / data_dir / "manifest.json"
cached = json.loads(dataset_manifest.read_text()) if dataset_manifest.exists() else {}
if "gpqa" in exp["benchmarks"] and ("gpqa" not in cached or not (WORK / data_dir / "gpqa.jsonl").is_file()):
    auth_check = "from huggingface_hub import HfApi; HfApi().whoami()"
    result = subprocess.run([PY, "-c", auth_check], env=env, capture_output=True)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("HF_TOKEN")
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            token = None
        if not token:
            token = getpass("Hugging Face read token for GPQA: ").strip()
        try:
            run_python("-c", "import sys; from huggingface_hub import login; "
                       "login(token=sys.stdin.read(), add_to_git_credential=False)",
                       input=token, text=True)
        finally:
            del token


run_python("-c", load_cfg + "from final_project.data import prepare; prepare(cfg)", str(settings))
storage["save_data"](WORK / data_dir, DRIVE / data_dir)
storage["copy_tree_once"](DRIVE / model_dir, WORK / model_dir)
if not (WORK / model_dir).exists():
    with tempfile.TemporaryDirectory(dir=WORK) as tmp:
        subprocess.run([PY, "-c", load_cfg + "from final_project.backend import convert; convert(cfg)", str(settings)],
                       cwd=tmp, env=env, check=True)
        (WORK / model_dir).parent.mkdir(parents=True, exist_ok=True)
        (Path(tmp) / model_dir).rename(WORK / model_dir)
run_python("-c", load_cfg + "from final_project.backend import verify_model; verify_model(cfg)", str(settings))
try:
    storage["copy_tree_once"](WORK / model_dir, DRIVE / model_dir)
except OSError:
    # Another evaluation runtime can finish publishing this model first.
    if not (DRIVE / model_dir / "config.json").is_file():
        raise
if manifest:
    for name, expected in manifest["weights"].items():
        if sha256(WORK / model_dir / name) != expected:
            raise ValueError(f"Model mismatch: {name}")
    for name, expected in manifest["datasets"].items():
        if sha256(WORK / data_dir / (name + ".jsonl")) != expected:
            raise ValueError(f"Dataset mismatch: {name}")

## Evaluate and resume
The worker reconstructs the training examples to verify the checkpoint identity, then uses the project’s LoRA prompts, generation and scoring functions. Each completed task gets one row in `results.jsonl`, including its model output.

In [ ]:
driver = r'''
import json, os, sys
from pathlib import Path
from final_project.config import digest, model_path, resume_results
from final_project.data import examples, tasks
from final_project.experiment import MLXBackend, Language, file_hash, generate_row, prefix
from final_project.report import report

settings, adapter, root = map(Path, sys.argv[1:4])
seed, n = map(int, sys.argv[4:6])
cfg = json.loads(settings.read_text())
lang = Language(seed)
rows, valid = examples(cfg, lang, seed)
marker = json.loads((adapter / "complete.json").read_text())
if marker["identity"] != digest({"cfg": cfg, "rows": rows[:n], "valid": valid, "seed": seed}):
    raise ValueError("Checkpoint belongs to a different training job. Check TARGET_MANIFEST, SEED and N.")
cfg["evaluation"]["sandbox"] = "linux-chroot"
batches = {b: tasks(cfg, b) for b in cfg["experiment"]["benchmarks"]}
expected = {f"lora/{n}/{seed}/{b}/{t['id']}" for b, batch in batches.items() for t in batch}
if len(expected) != sum(map(len, batches.values())):
    raise ValueError("Duplicate task IDs in benchmark data")
weights = model_path(cfg)
inputs = {"revision": sys.argv[6], "config": cfg, "seed": seed, "n": n, "checkpoint": marker,
          "adapter_config": file_hash(adapter / "adapter_config.json"),
          "model": {p.name: file_hash(p) for p in sorted(weights.iterdir()) if p.is_file()},
          "tasks": digest(batches), "plan": {"evaluations": len(expected)}}
root.mkdir(parents=True, exist_ok=True)
manifest_path = root / "manifest.json"
log = root / "results.jsonl"
if manifest_path.exists():
    if json.loads(manifest_path.read_text()) != inputs:
        raise ValueError("Saved evaluation inputs differ. Use a separate output folder.")
elif log.exists():
    raise ValueError("Saved results have no manifest to verify their inputs.")
else:
    tmp = root / ".manifest.tmp"
    tmp.write_text(json.dumps(inputs, indent=2))
    tmp.replace(manifest_path)
previous = resume_results(log)
done = {r["key"] for r in previous}
if len(done) != len(previous) or not done <= expected:
    raise ValueError("Unexpected or duplicate saved evaluation keys")
print(f"Restored {len(done)}/{len(expected)} evaluations.", flush=True)
backend = None
for benchmark, batch in batches.items():
    pending = [t for t in batch if f"lora/{n}/{seed}/{benchmark}/{t['id']}" not in done]
    if not pending:
        continue
    if backend is None:
        backend = MLXBackend(cfg, adapter)
    messages = prefix(lang, "lora", [], benchmark.endswith("synthetic"))
    backend.set_prefix(messages)
    for task in pending:
        row = generate_row(backend, cfg, "lora", n, seed, benchmark, lang, messages, [], task)
        row["generation_workers"] = 1
        with log.open("a") as handle:
            handle.write(json.dumps(row) + "\n")
            handle.flush()
            os.fsync(handle.fileno())
        done.add(row["key"])
        print(f"{len(done)}/{len(expected)}  {row['key']}  {row['status']}", flush=True)
report(root)
'''
output = Path("lora-evaluations") / exp["name"] / label
local, saved = WORK / output, DRIVE / output
storage["copy_tree_once"](saved, local)
backup = storage["RunBackup"](local, saved)
backup.run([PY, "-c", driver, str(settings), str(adapter), str(local), str(SEED), str(N), REVISION], env, WORK)
print("Drive results:", saved)
from IPython.display import Markdown, display
display(Markdown((local / "report.md").read_text()))